In [23]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import make_column_selector, make_column_transformer
from sklearn.impute import SimpleImputer

import os
os.chdir('E:\Backup Files Abhinav_Satyam\Practical Machine Learning\Sanjay Sir Git Repo\Datasets')


In [24]:
loan = pd.read_csv('loan.csv',index_col=0)
loan

,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
Loan_ID,,,,,,,,,,,,
LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y
...,...,...,...,...,...,...,...,...,...,...,...,...
LP002978,Female,No,0,Graduate,No,2900,0.0,71.0,360.0,1.0,Rural,Y
LP002979,Male,Yes,3+,Graduate,No,4106,0.0,40.0,180.0,1.0,Rural,Y
LP002983,Male,Yes,1,Graduate,No,8072,240.0,253.0,360.0,1.0,Urban,Y


In [26]:
loan.isna().sum()

Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64

In [20]:
imputer = SimpleImputer(strategy='most_frequent').set_output(transform='pandas')
# loan = imputer.fit_transform(loan)

In [41]:
loan = pd.read_csv('loan.csv',index_col=0)
X , y = loan.drop('Loan_Status' , axis = 1), loan['Loan_Status']


imp_cat = SimpleImputer(strategy='most_frequent').set_output(transform='pandas')
imp_num = SimpleImputer().set_output(transform='pandas')

ct_imp = make_column_transformer(
                            (imp_cat,           make_column_selector(dtype_include = object)),
                            (imp_num, make_column_selector(dtype_include =['int64','float64'])),
                             verbose_feature_names_out = False) 

ct_imp = ct_imp.set_output(transform = 'pandas')

X_trans = ct_imp.fit_transform(X)
X_trans.isnull().sum().sum()

np.int64(0)

In [42]:
ohc = OneHotEncoder(sparse_output = False, drop = 'first').set_output(transform = 'pandas')
ct_ohe = make_column_transformer(
                            (ohc,           make_column_selector(dtype_include = object)),
                            ("passthrough", make_column_selector(dtype_include =['int64','float64'])),
                             verbose_feature_names_out = False)

ct_ohe = ct_ohe.set_output(transform='pandas')
X_trans_ohe = ct_ohe.fit_transform(X_trans)
X_trans_ohe.dtypes

Gender_Male                float64
Married_Yes                float64
Dependents_1               float64
Dependents_2               float64
Dependents_3+              float64
Education_Not Graduate     float64
Self_Employed_Yes          float64
Property_Area_Semiurban    float64
Property_Area_Urban        float64
ApplicantIncome            float64
CoapplicantIncome          float64
LoanAmount                 float64
Loan_Amount_Term           float64
Credit_History             float64
dtype: object

In [44]:
loan = pd.read_csv('loan.csv',index_col=0)
X , y = loan.drop('Loan_Status' , axis = 1), loan['Loan_Status']
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.3, random_state=26, stratify=y)



ohe = OneHotEncoder(sparse_output = False, drop = 'first').set_output(transform = 'pandas')
imputer_cat = SimpleImputer(strategy='most_frequent').set_output(transform='pandas')
imputer_num = SimpleImputer(strategy='mean').set_output(transform='pandas')
#2 imputer one for categorical for numerical
#and 2 ct, 1 for imputer and one for ohe
                        
ct_impute = make_column_transformer((imputer_cat,make_column_selector(dtype_include=object)),
                            (imputer_num,make_column_selector(dtype_include=['int64','float64'])),
                            verbose_feature_names_out=False).set_output(transform='pandas')

ct_ohe = make_column_transformer((ohe,make_column_selector(dtype_include=object)),
                            ('passthrough',make_column_selector(dtype_include=['int64','float64'])),
                            verbose_feature_names_out=False).set_output(transform='pandas')

scaler = StandardScaler().set_output(transform='pandas')



ks = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]
scores=[]
for k in ks:
    knn = KNeighborsClassifier(n_neighbors=k)
    pipeKNN = Pipeline([('ct_impute',ct_impute),('ct',ct_ohe),('Scaler',scaler),('KNN',knn)])
    pipeKNN.fit(X_train,y_train)
    y_pred = pipeKNN.predict(X_test)
    scores.append([k,accuracy_score(y_test,y_pred)])
df_scores = pd.DataFrame(scores,columns=['K','score'])    
df_scores.sort_values('score',ascending=False)


,K,score
6,7,0.843243
5,6,0.843243
7,8,0.843243
10,11,0.837838
9,10,0.837838
8,9,0.837838
12,13,0.837838
13,14,0.837838
4,5,0.832432
11,12,0.832432
